# Earnalism narration job `narration-bc4edba3`

**Prepared, not run.** Input zip SHA256 `7352b8263bba031c66b42fbf6bd502f3f8d5ed89eb218f482d29ca00136761ae`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/prayashchitta/narration-bc4edba3/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 9625 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'narration-bc4edba3'
TITLE = 'prayashchitta'
KIND = 'narration'
INPUT_ZIP_SHA256 = '7352b8263bba031c66b42fbf6bd502f3f8d5ed89eb218f482d29ca00136761ae'
MODEL_REPO = 'ai4bharat/indic-parler-tts'
MODEL_REVISION = '7b527af5ee8ed1f9a28d80b19703ed9bb8ba10ca'
STEPS = 1
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/prayashchitta/narration-bc4edba3'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 9625
MANIFEST_RESERVE_S = 60
PACKAGES = ('parler_tts', 'transformers', 'torch', 'soundfile')


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Narration: one DRY WAV per voice unit with Indic Parler-TTS (revision pinned at build time).
# No pause is written here: eav.film.narration_track inserts every pause exactly once (R8).
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/parler-tts.git", "soundfile"], check=True)
import soundfile as sf, torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ParlerTTSForConditionalGeneration.from_pretrained(
    MODEL_REPO, revision=MODEL_REVISION).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, revision=MODEL_REVISION)
desc_tokenizer = AutoTokenizer.from_pretrained(
    model.config.text_encoder._name_or_path)
GENERATION = dict(JOB.get("generation", {}))  # pinned roster preset settings (R9)
if GENERATION and "do_sample" not in GENERATION:
    GENERATION["do_sample"] = True  # a temperature only applies when sampling
MODEL_INFO = {"repo": MODEL_REPO, "revision": MODEL_REVISION, "generation": GENERATION}

rate = model.config.sampling_rate
EXPECTED = [unit["unit_key"] + ".wav" for unit in JOB["units"]]
last = 0.0  # seconds the previous unit took; the next starts only if it fits (R25)
for unit in JOB["units"]:
    if not may_start(last):
        break
    started = time.time()
    d = desc_tokenizer(unit["description"], return_tensors="pt").to(device)
    p = tokenizer(unit["text"], return_tensors="pt").to(device)
    audio = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                           prompt_input_ids=p.input_ids,
                           prompt_attention_mask=p.attention_mask, **GENERATION)
    wav = audio.cpu().numpy().squeeze().astype("float32")
    sf.write(str(OUT_DIR / (unit["unit_key"] + ".wav")), wav, rate)
    last = time.time() - started
    print("wrote", unit["unit_key"], "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip"):
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
